### Cellule 1 — Transformations

In [1]:
import torchvision.transforms as transforms

train_transform = transforms.Compose([
 transforms.Resize(256),
 transforms.RandomCrop(224),
 transforms.RandomHorizontalFlip(),
 transforms.ColorJitter(brightness=0.2, contrast=0.2),
 transforms.ToTensor(),
 transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_transform = transforms.Compose([
 transforms.Resize(256),
 transforms.CenterCrop(224),
 transforms.ToTensor(),
 transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])


### Cellule 2 — Chargement des datasets

In [4]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from pathlib import Path

# Définition des chemins
DATA_DIR = Path('../data/raw/Rock Data')
TRAIN_DIR = DATA_DIR / 'train'
VALID_DIR = DATA_DIR / 'valid'
TEST_DIR = DATA_DIR / 'test'

# Création des datasets avec les transformations de la Phase 2
train_ds = ImageFolder(TRAIN_DIR, transform=train_transform)
valid_ds = ImageFolder(VALID_DIR, transform=val_transform)
test_ds = ImageFolder(TEST_DIR, transform=val_transform)

# Création des DataLoaders (num_workers=0 pour stabilité sous Windows)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
valid_loader = DataLoader(valid_ds, batch_size=32, shuffle=False, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

print(f"Datasets chargés avec succès !")
print(f"Classes détectées : {train_ds.classes}")

Datasets chargés avec succès !
Classes détectées : ['Basalt', 'Clay', 'Conglomerate', 'Diatomite', 'Shale-(Mudstone)', 'Siliceous-sinter', 'chert', 'gypsum', 'olivine-basalt']


### Cellule 3 — Vérification

In [5]:
# Vérifier que tout fonctionne
images, labels = next(iter(train_loader))
print(f'Batch shape : {images.shape}') # doit être [32, 3, 224, 224]
print(f'Classes : {train_ds.classes}')

Batch shape : torch.Size([32, 3, 224, 224])
Classes : ['Basalt', 'Clay', 'Conglomerate', 'Diatomite', 'Shale-(Mudstone)', 'Siliceous-sinter', 'chert', 'gypsum', 'olivine-basalt']


### Résumé


### 1. Création du Pipeline de Transformation (La "Cuisine")
Nous avons mis en place deux recettes distinctes pour préparer mathématiquement tes images avant de les donner au modèle ResNet-50 :

* **Le Pipeline d'Entraînement (`train_transform`) :** * Nous avons intégré de la **Data Augmentation** (`RandomCrop(224)`, `RandomHorizontalFlip()`, `ColorJitter`). 
    * *L'objectif :* Forcer le modèle à se concentrer sur la texture des roches et le rendre robuste face aux défauts identifiés en Phase 1 (les mains sur les roches, les filigranes "alamy", les variations de luminosité).
* **Le Pipeline d'Évaluation (`val_transform`) :** * Nous avons opté pour une approche stricte et reproductible (`CenterCrop(224)`) sans aucun hasard, afin de garantir que l'examen du modèle soit toujours juste et identique.
* **La Normalisation :** * Les deux pipelines se terminent par `ToTensor()` et `Normalize()`. Nous avons calqué la distribution de tes couleurs sur celle d'ImageNet (moyenne et écart-type) pour que le modèle pré-entraîné ne soit pas "aveuglé" par de nouvelles couleurs brutes.

### 2. Automatisation du Chargement (La "Logistique")
Plutôt que de charger 4000 images en mémoire vive (ce qui ferait planter ton ordinateur), nous avons utilisé les outils natifs de PyTorch :

* **`ImageFolder` :** Il a scanné tes dossiers, identifié tes 9 classes par ordre alphabétique, et prépare les images *à la volée* (uniquement quand on en a besoin).
* **`DataLoader` :** Il agit comme un convoyeur. Il rassemble tes images par paquets de 32 (`batch_size=32`). Il mélange activement les paquets d'entraînement (`shuffle=True`) pour éviter que le modèle n'apprenne l'ordre des roches par cœur, et laisse les paquets de test dans l'ordre (`shuffle=False`).

### 3. Sécurisation de l'Environnement Local (L'astuce Windows)
* Nous avons évité un piège majeur de PyTorch sous Windows en fixant **`num_workers=0`**. Cela désactive le multiprocessing dans le Jupyter Notebook, t'évitant ainsi des plantages silencieux et garantissant la stabilité totale de ton code sur ta machine.

### 4. Le "Sanity Check" (La Validation Finale)
* Nous avons extrait un lot test pour vérifier la "tuyauterie". 
* L'affichage de la dimension **`[32, 3, 224, 224]`** nous a confirmé que ton lot contient bien 32 images, en 3 couleurs (RGB), parfaitement découpées en 224x224 pixels. La géométrie est impeccable et prête à être ingérée par ResNet-50.

---

**Le Bilan :** Ta base de données brute est maintenant devenue un flux de données standardisé, nettoyé et optimisé. Ton environnement local a rempli son rôle à 100%.

